In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle
import os

In [2]:

# read data 
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-01.parquet')

df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
df['duration'] = df.duration.apply(lambda td: td.total_seconds() / 60)

df = df[(df.duration >= 1) & (df.duration <= 60)].copy()
df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)

In [3]:
categorical = ['PU_DO']
numerical = ['trip_distance']

train_dicts = df[categorical + numerical].to_dict(orient='records')
#encoding 
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

# the target which is trip duration 
y_train = df['duration'].values

In [4]:
# train model 
model = LinearRegression()
model.fit(X_train, y_train)

# prediction
y_pred = model.predict(X_train)

mse = mean_squared_error(y_train, y_pred)
rmse = mse ** 0.5  
mae = mean_absolute_error(y_train, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

RMSE: 4.8197
MAE: 3.3130


In [5]:
os.makedirs('../models', exist_ok=True)

with open('../models/baseline.pkl', 'wb') as f:
    pickle.dump((dv, model), f)
    
print("Model saved successfully!")

Model saved successfully!
